# Manual Media-Source Token Diagnostics

Paste one rebuilt content item as JSON, run the media-source model, and inspect entity and token-level diagnostics.


## Setup

Run this notebook from an environment where `impresso-pipelines[mediasources]` is installed. For local development, the cells below also add the repository root and the sibling `impresso-pipelines` checkout to `sys.path` when present.


In [ ]:
from __future__ import annotations

import html
import inspect
import json
import sys
from pathlib import Path
from typing import Any

try:
    from IPython.display import HTML, display
except ImportError:
    class HTML(str):
        pass

    def display(value: Any) -> None:
        print(value)

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'lib').exists() else NOTEBOOK_DIR.parent
SIBLING_PIPELINES = REPO_ROOT.parent / 'impresso-pipelines'

for path in [REPO_ROOT, SIBLING_PIPELINES]:
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

from impresso_pipelines.mediasources import MediaSourcesPipeline


/Users/siclemat/pj/2025/impresso/impresso-mediasources-cookbook/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Paste A Rebuilt Content Item

Replace `CONTENT_ITEM_JSON` with one rebuilt content-item JSON object. The notebook reads text from `ft` and uses `ci_id`, `id`, or `c_id` as the display identifier.


In [ ]:
CONTENT_ITEM_JSON = r'''
{
  "ci_id": "manual-example",
  "date": "1951-01-01",
  "ft": "Reuters reported the news. BBC broadcast the statement. Radio Prague announced the programme."
}
'''

content_item = json.loads(CONTENT_ITEM_JSON)
text = content_item.get('ft') or ''
content_id = content_item.get('ci_id') or content_item.get('id') or content_item.get('c_id') or '<missing id>'
publication_date = content_item.get('date') or content_item.get('d') or content_item.get('year') or content_item.get('publication_date')

if not isinstance(text, str) or not text.strip():
    raise ValueError('The content item must contain non-empty full text in the ft property.')

print(f'id: {content_id}')
print(f'publication_date: {publication_date}')
print(f'characters: {len(text)}')


## Load The Pipeline

Set `LOCAL_FILES_ONLY = False` if the model is not already cached and the notebook environment has Hugging Face network access.


In [ ]:
MODEL_ID = 'impresso-project/mmbert-impresso-mediasources-ner'
REVISION = 'v2.0.0'
DEVICE = -1  # CPU. Use 0 for the first CUDA/MPS-supported accelerator when available.
BATCH_SIZE = 32
LOCAL_FILES_ONLY = True
MIN_SCORE = None
FILTER_ANACHRONISTIC = False

pipe = MediaSourcesPipeline(
    model=MODEL_ID,
    revision=REVISION,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    min_score=MIN_SCORE,
    local_files_only=LOCAL_FILES_ONLY,
)


## Run Diagnostics


In [ ]:
call_kwargs: dict[str, Any] = {'diagnostics': True}
signature = inspect.signature(pipe.__call__)
if 'publication_date' in signature.parameters:
    call_kwargs['publication_date'] = publication_date
if 'filter_anachronistic' in signature.parameters:
    call_kwargs['filter_anachronistic'] = FILTER_ANACHRONISTIC

result = pipe(text, **call_kwargs)
result


## Display Helpers


In [ ]:
def html_table(rows: list[dict[str, Any]], columns: list[str]) -> HTML:
    if not rows:
        return HTML('<p><em>No rows.</em></p>')
    header = ''.join(f'<th>{html.escape(column)}</th>' for column in columns)
    body_rows = []
    for row in rows:
        cells = []
        for column in columns:
            value = row.get(column, '')
            if isinstance(value, float):
                value = f'{value:.6f}'
            cells.append(f'<td>{html.escape(str(value))}</td>')
        body_rows.append('<tr>' + ''.join(cells) + '</tr>')
    style = '''
    <style>
      table.mediasources { border-collapse: collapse; font-size: 13px; }
      table.mediasources th, table.mediasources td { border: 1px solid #ddd; padding: 4px 7px; vertical-align: top; }
      table.mediasources th { background: #f3f4f6; text-align: left; }
      table.mediasources tr:nth-child(even) { background: #fafafa; }
      .ms-entity { background: #fff3b0; border-bottom: 2px solid #d97706; padding: 0 2px; }
      .ms-text { line-height: 1.8; white-space: pre-wrap; }
    </style>
    '''
    return HTML(style + '<table class="mediasources"><thead><tr>' + header + '</tr></thead><tbody>' + ''.join(body_rows) + '</tbody></table>')

def token_rows(result: dict[str, Any]) -> list[dict[str, Any]]:
    tokens = result.get('tokens', [])
    starts = result.get('token_start_offsets', [])
    stops = result.get('token_end_offsets', [])
    labels = result.get('token_labels', [])
    scores = result.get('token_scores', [])
    return [
        {'i': i, 'token': token, 'start': starts[i], 'stop': stops[i], 'label': labels[i], 'score': scores[i]}
        for i, token in enumerate(tokens)
    ]

def highlighted_text(text: str, entities: list[dict[str, Any]]) -> HTML:
    pieces = []
    cursor = 0
    for entity in sorted(entities, key=lambda item: (item.get('start', 0), item.get('stop', 0))):
        start = int(entity.get('start', cursor))
        stop = int(entity.get('stop', start))
        if start < cursor or stop < start:
            continue
        pieces.append(html.escape(text[cursor:start]))
        label = html.escape(str(entity.get('label', '')))
        qid = html.escape(str(entity.get('wkdata_qid') or ''))
        title = f'{label} {qid}'.strip()
        pieces.append(f'<span class="ms-entity" title="{title}">{html.escape(text[start:stop])}</span>')
        cursor = stop
    pieces.append(html.escape(text[cursor:]))
    return HTML('<div class="ms-text">' + ''.join(pieces) + '</div>')


## Entities


In [ ]:
entities = result.get('entities', [])
display(html_table(entities, ['surface', 'label', 'wkdata_qid', 'start', 'stop', 'score']))
display(highlighted_text(text, entities))


## Summary


In [ ]:
display(html_table(result.get('summary', []), ['uid', 'wkdata_qid', 'score']))


## Token Diagnostics


In [ ]:
display(html_table(token_rows(result), ['i', 'token', 'start', 'stop', 'label', 'score']))


## JSON Result


In [ ]:
print(json.dumps(result, ensure_ascii=False, indent=2))
